# Regex script-gate + real fastText LID -- ninth method

Same script-gating shell as `08_regex_baseline.ipynb`, but the decision
*inside* each group now comes from Facebook's actual pretrained
**fastText language-identification model** (`lid.176`, 176 languages,
trained on Wikipedia + Tatoeba + SETimes) instead of a hand-curated
marker-word list -- this is the real, off-the-shelf tool, not this
project's own from-scratch "fastText-style" classifier
(`03_fasttext_classifier.ipynb`, which trains a small hashed-n-gram
model on this project's own labeled data; this notebook trains nothing
at all).

**Getting the real `fasttext` package running here needed two fixes**,
both applied to a local build/install, not worked around --
transparency over silently reaching for a substitute:
1. **A genuine MSVC compile bug**: `fasttext`'s C++ pybind wrapper uses
   the POSIX `ssize_t` type, which doesn't exist on Windows/MSVC. Fixed
   by patching a one-time local build of `fasttext-wheel==0.9.2`'s
   `fasttext_pybind.cc` to typedef it from `SSIZE_T` (`BaseTsd.h`) under
   `_MSC_VER`, then building from a short path (`C:\ft_build`) --  the
   *un*patched build's *next* error, `cl.exe`'s cryptic "Cannot open
   compiler generated file", turned out to be `MAX_PATH` (260 chars)
   overflow from the original build directory's long, deeply-nested path,
   not a real compile problem.
2. **A NumPy 2.0 compatibility bug**: the installed package's own
   `FastText.py` calls `np.array(x, copy=False)`, which NumPy 2.0 made
   stricter (raises instead of silently copying when a true zero-copy
   view isn't possible) -- patched to `np.asarray(x)` in the installed
   package, the officially-recommended fix per NumPy's own 2.0 migration
   guide.

`lid.176.ftz` (the quantized/compressed ~938KB model, not the full
~126MB `.bin`) downloaded directly from
`dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz` to
`models/fasttext_lid/lid.176.ftz` -- not tracked in git (see
`.gitignore`), re-downloadable via the same URL any time.

**The rule, per spec** -- same script-gating as every other method here
(`code_switch`/`other` via the deterministic regex, unchanged):
- **`latin` group**: `lid.176` predicts among all 176 languages; if its
  top prediction is `fr` -> `french`, if `en` -> `english`, **anything
  else -> `arabize`** (`lid.176` has no Arabizi/Romanized-Arabic class
  at all, so this fallback is doing real work, not just mopping up rare
  edge cases -- see the results below).
- **`arabic` group**: if `lid.176`'s top prediction is `ar` -> `msa`,
  **anything else -> `darija`** (again, `lid.176` has no "Arabic
  dialect" classes -- MSA-vs-Darija was never a distinction it was
  trained to make, only "is this Arabic script at all").

Runs in the **base** Python environment -- `fasttext`'s C extension was
built there, `torch` is not needed for this notebook at all.


In [1]:
import json
import re
import sys
from pathlib import Path

import fasttext

fasttext.FastText.eprint = lambda *a, **k: None  # silence fastText's own load-time warning spam

ROOT = Path.cwd().resolve().parents[1]  # project root, when running from Dialect_Identification/notebooks/
DATA_DIR = ROOT / "Dialect_Identification" / "data"
MODEL_PATH = ROOT / "Dialect_Identification" / "models" / "fasttext_lid" / "lid.176.ftz"

lid_model = fasttext.load_model(str(MODEL_PATH))
print(f"Loaded {MODEL_PATH.name} -- {len(lid_model.get_labels())} languages")


Loaded lid.176.ftz -- 176 languages


## 1. Script-gating (same convention as every other method here)

In [2]:
_MENTION_WITH_FRAGMENT_RE = re.compile(r"\[MENTION\](\s*-[^\s]{1,10})?")
_URL_RE = re.compile(r"\[URL\]")
_INLINE_WHITESPACE_RE = re.compile(r"[ \t]+")
_ALL_WHITESPACE_RE = re.compile(r"\s+")
_ARABIC_RE = re.compile(r"[\u0600-\u06ff\u0750-\u077f\u08a0-\u08ff\ufb50-\ufdff\ufe70-\ufeff]")
_LATIN_RE = re.compile(r"[a-zA-Z\u00c0-\u024f]")

ARABIC_CLASSES = ["msa", "darija"]
LATIN_CLASSES = ["arabize", "french", "english"]
GROUP_CLASSES = {"arabic": ARABIC_CLASSES, "latin": LATIN_CLASSES}

MAX_CHARS = 500  # same truncation convention as label_dataset.py's MAX_PROMPT_CHARS


def clean_for_classification(text: str) -> str:
    text = _MENTION_WITH_FRAGMENT_RE.sub("", text)
    text = _URL_RE.sub("", text)
    return _INLINE_WHITESPACE_RE.sub(" ", text).strip()


def script_of(text: str) -> str:
    has_ar = bool(_ARABIC_RE.search(text))
    has_lat = bool(_LATIN_RE.search(text))
    if has_ar and has_lat:
        return "mixed"
    if has_ar:
        return "arabic"
    if has_lat:
        return "latin"
    return "other"


def load_group_rows(split: str, group: str) -> list[dict]:
    rows = []
    with (DATA_DIR / f"{split}.jsonl").open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            cleaned = clean_for_classification(r["text"])
            if script_of(cleaned) != group:
                continue
            rows.append({"id": r["id"], "text": r["text"], "label": r["label"]})
    return rows


## 2. Classification rule

`fasttext.predict()` requires a single line (no `\n`) -- `_ALL_WHITESPACE_RE`
collapses every whitespace run (including real newlines, which
`clean_for_classification` deliberately leaves alone elsewhere in this
project) into one space, just for this model call.

In [3]:
def fasttext_top_label(text: str) -> str:
    single_line = _ALL_WHITESPACE_RE.sub(" ", text[:MAX_CHARS]).strip()
    if not single_line:
        return ""
    labels, _ = lid_model.predict(single_line, k=1)
    return labels[0].removeprefix("__label__")


def classify_latin(text: str) -> str:
    lang = fasttext_top_label(text)
    if lang == "fr":
        return "french"
    if lang == "en":
        return "english"
    return "arabize"


def classify_arabic(text: str) -> str:
    lang = fasttext_top_label(text)
    return "msa" if lang == "ar" else "darija"


def classify(text: str, group: str) -> str:
    return classify_latin(text) if group == "latin" else classify_arabic(text)


## 3. Evaluate on `test.jsonl`

In [4]:
all_results = {}
for group in ("arabic", "latin"):
    test_rows = load_group_rows("test", group)
    y_true = [r["label"] for r in test_rows]
    y_pred = [classify(r["text"], group) for r in test_rows]
    acc = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
    all_results[group] = dict(rows=test_rows, y_true=y_true, y_pred=y_pred, accuracy=acc)

    print(f"\n{group}: accuracy={acc:.4f} ({sum(t == p for t, p in zip(y_true, y_pred))}/{len(y_true)})")
    print("Per-class accuracy:")
    for cname in GROUP_CLASSES[group]:
        idx = [i for i, t in enumerate(y_true) if t == cname]
        n_correct = sum(y_pred[i] == cname for i in idx)
        print(f"  {cname:<10} {n_correct:>4}/{len(idx):<4} acc={n_correct / len(idx) if idx else float('nan'):.3f}")



arabic: accuracy=0.3399 (566/1665)
Per-class accuracy:
  msa         538/542  acc=0.993
  darija       28/1123 acc=0.025



latin: accuracy=0.6296 (719/1142)
Per-class accuracy:
  arabize     281/621  acc=0.452
  french      353/415  acc=0.851
  english      85/106  acc=0.802


In [5]:
def print_confusion(classes, y_true, y_pred, title):
    n = len(classes)
    idx = {c: i for i, c in enumerate(classes)}
    mat = [[0] * n for _ in range(n)]
    for t, p in zip(y_true, y_pred):
        mat[idx[t]][idx[p]] += 1
    print(f"\n{title} confusion matrix (rows=true, cols=predicted):")
    print("true\\pred" + "".join(f"{c:>12}" for c in classes))
    for i, c in enumerate(classes):
        print(f"{c:<10}" + "".join(f"{mat[i][j]:>12}" for j in range(n)))


for group, res in all_results.items():
    print_confusion(GROUP_CLASSES[group], res["y_true"], res["y_pred"], group)



arabic confusion matrix (rows=true, cols=predicted):
true\pred         msa      darija
msa                538           4
darija            1095          28

latin confusion matrix (rows=true, cols=predicted):
true\pred     arabize      french     english
arabize            281         137         203
french              27         353          35
english             20           1          85


## 4. What `lid.176` actually predicted, before the fallback rule collapsed it

Useful diagnostic specific to this method: since `lid.176` knows 176
languages but this task only keeps 2-3 of them per group, seeing the
*raw* top-language distribution shows how much real signal the fallback
rule is absorbing (e.g. how often Arabizi gets called Finnish/Somali/
Swahili/etc. by a model that's never seen romanized Arabic as its own
class).

In [6]:
from collections import Counter

for group in ("arabic", "latin"):
    raw_langs = Counter(fasttext_top_label(r["text"]) for r in all_results[group]["rows"])
    print(f"\n{group}: top-10 raw lid.176 predictions (before the fallback rule):")
    for lang, count in raw_langs.most_common(10):
        print(f"  {lang:<6} {count:>5}  ({count / len(all_results[group]['rows']):.1%})")



arabic: top-10 raw lid.176 predictions (before the fallback rule):
  ar      1633  (98.1%)
  fa        21  (1.3%)
  arz        3  (0.2%)
  zh         3  (0.2%)
  ja         1  (0.1%)
  ug         1  (0.1%)
  ur         1  (0.1%)
  ru         1  (0.1%)
  pnb        1  (0.1%)

latin: top-10 raw lid.176 predictions (before the fallback rule):
  fr       491  (43.0%)
  en       323  (28.3%)
  es        28  (2.5%)
  fi        27  (2.4%)
  de        23  (2.0%)
  it        20  (1.8%)
  pt        18  (1.6%)
  nl        16  (1.4%)
  pl        15  (1.3%)
  sw        14  (1.2%)


## 5. Misclassified examples

In [7]:
import random

random.seed(0)

for group, res in all_results.items():
    wrong = [
        (r["text"], t, p) for r, t, p in zip(res["rows"], res["y_true"], res["y_pred"]) if t != p
    ]
    print(f"\n=== {group}: {len(wrong)}/{len(res['rows'])} misclassified ({len(wrong) / len(res['rows']):.1%}) ===")
    sample = random.sample(wrong, min(12, len(wrong)))
    for text, true_label, pred_label in sample:
        snippet = text[:140] + ("..." if len(text) > 140 else "")
        print(f"\n  true={true_label}  pred={pred_label}  (raw lid.176: {fasttext_top_label(text)!r})")
        print(f"  {snippet!r}")



=== arabic: 1099/1665 misclassified (66.0%) ===

  true=darija  pred=msa  (raw lid.176: 'ar')
  'المسلسلات و الافلام تاع العالم كاين حتى اللي قصصهم على الفضاء غير تاع دزاير تلعبوها تموتو على الواقع و كي يديرو تاع الواقع كامل على لا دروڨ ...'

  true=darija  pred=msa  (raw lid.176: 'ar')
  'ما أكثرهم مثلها في هاد الزمان'

  true=darija  pred=msa  (raw lid.176: 'ar')
  'يعني باينة مراحش تراجعي منا باسكو مكاش مصطلحات افهمي برك منا بصح راجعي من كراسك ولا كتاب خارجي'

  true=darija  pred=msa  (raw lid.176: 'ar')
  'بصح يكسيها ويشريلها ذهب راهي كيف كيف'

  true=darija  pred=msa  (raw lid.176: 'ar')
  'عبد الله\nالسلام عليكم\n*ما قول ضيفنا في\nمن تعلم بعض العلم في البرمجيات واصبح يقرصن في المخلوقات ؟\nهل التكنولوجيا جعلتنا عبيدا عنده\nا ؟\nأمثال ...'

  true=darija  pred=msa  (raw lid.176: 'ar')
  'انا تاني أعطاني 400 الف بصح شديتهم يا اختي 😂😂'

  true=darija  pred=msa  (raw lid.176: 'ar')
  'بيسة\nخآلقي وآحد أحد\nلآ اله الآ ألله\nمررت بالعديد من الظروف لكن أملي بالله لآ يعوض و لآ يخيب\nونعم

## 6. Compared against the marker-word baseline (`08_regex_baseline.ipynb`)

In [8]:
print(f"{'method':<35}{'arabic acc':>12}{'latin acc':>12}")
print(f"{'regex marker-word (08)':<35}{0.699:>12.3f}{0.748:>12.3f}")
print(f"{'regex + real fastText lid.176 (09)':<35}{all_results['arabic']['accuracy']:>12.3f}{all_results['latin']['accuracy']:>12.3f}")


method                               arabic acc   latin acc
regex marker-word (08)                    0.699       0.748
regex + real fastText lid.176 (09)        0.340       0.630
